# CA1 — Meal Calorie Helper (T17)

**CSE476 Agentic AI and Intelligent Automation — CA1**

This notebook runs the real Groq-backed agent (the same code used by
`api_server.py` and `main.py`, imported from the `app` package — nothing
here is re-implemented or faked for the demo) on 3 example goals and
prints the full plan-act trace for each, so the multi-step tool-calling
loop and memory usage are directly visible.

Official tools demonstrated: `lookup_calories(food)`, `add_meal(food)`.
Extension tool also demonstrated: `remove_meal(food)`.

**Requires a real `GROQ_API_KEY`** in `.env` (or the environment) to
actually call the model — this notebook makes live Groq calls, it does
not mock them (unlike the unit tests in `tests/`, which do mock Groq).

In [ ]:
import sys, os, json, uuid
sys.path.insert(0, os.path.abspath(".."))

from app.agent import run_agent, analyze_food_image
from app.memory import memory_store

assert os.getenv("GROQ_API_KEY"), "Set GROQ_API_KEY in your environment or .env before running this notebook."

session_id = str(uuid.uuid4())
print("Demo session:", session_id)

In [ ]:
def show_trace(result):
    """Pretty-print the plan-act trace so every step is visible:
    LLM decision -> tool call -> tool result -> memory -> final answer."""
    for ev in result["trace"]:
        line = f"[step {ev['step']}] {ev['action']:<14} decision_source={ev['decision_source']}"
        if ev["action"] == "tool_call":
            line += f"  tool={ev['tool']}  args={ev['args']}"
        elif ev["action"] == "tool_result":
            line += f"  tool={ev['tool']}  result={ev['result']}"
        elif ev["action"] == "read_memory":
            line += f"  memory={ev['result']}"
        elif ev["action"] == "final_answer":
            line += f"  text={ev['text'][:200]!r}"
        print(line)
    print("\n--- FINAL ANSWER ---")
    print(result["reply"])
    print("\n--- MEMORY STATE AFTER THIS TURN ---")
    print(json.dumps(result["state"], indent=2))

## Demo 1 — Meal tracking (multi-step tool loop)

Goal: *"My calorie budget is 2000 calories. I had oatmeal and a banana
for breakfast."*

Expect to see: `lookup_calories("oatmeal")` -> `lookup_calories("banana")`
-> `add_meal("oatmeal", ...)` -> `add_meal("banana", ...)` -> final
answer. Each tool result is fed back to the LLM before it decides the
next action — this is the real plan-act loop, not a scripted sequence.

In [ ]:
result_1 = run_agent(
    session_id,
    "My calorie budget is 2000 calories. I had oatmeal and a banana for breakfast.",
)
show_trace(result_1)

## Demo 2 — Memory + snack decision + structured recommendation

Goal: *"What can I have for my next meal, and can you suggest 3 options
that fit my remaining calories?"*

Expect to see: a `read_memory` step showing the oatmeal/banana logged in
Demo 1 (proving memory persists across turns in this session), the agent
reasoning about the *remaining* budget, and a **structured Markdown**
response (numbered list, bolded dish names, a "Recommended Pick"
section) rather than one long paragraph.

In [ ]:
result_2 = run_agent(
    session_id,
    "What can I have for my next meal, and can you suggest 3 options that fit my remaining calories?",
)
show_trace(result_2)

## Demo 3 — Meal removal + an unfamiliar food (no hard failure)

Two short goals in sequence, both against the SAME session/memory:

1. *"Remove the banana I logged earlier."* — demonstrates `remove_meal`,
   a real memory read, memory mutation, and recalculated totals.
2. *"I had dragon fruit cheesecake for lunch, please log it."* — this
   food is NOT in the local `lookup_calories` table. Expect the agent to
   call `lookup_calories`, see `found=False`, and then still call
   `add_meal` with its own clearly-labelled **estimated** calorie value
   instead of refusing the request outright.

In [ ]:
result_3a = run_agent(session_id, "Remove the banana I logged earlier.")
show_trace(result_3a)

In [ ]:
result_3b = run_agent(session_id, "I had dragon fruit cheesecake for lunch, please log it.")
show_trace(result_3b)

## Bonus — scope rejection

A clearly irrelevant request should be politely refused, without calling
any meal/calorie tool.

In [ ]:
result_4 = run_agent(session_id, "What is the capital of France?")
show_trace(result_4)

## Bonus — image analysis (FEATURE 3)

Optional cell: point `IMAGE_PATH` at a real food photo on disk to see the
vision-model call. This uses a **separate** Groq vision-capable model
(`GROQ_VISION_MODEL`) from the main text agent, and always returns an
approximate, clearly-labelled estimate — never an exact calorie claim
from a photo alone. Skipped automatically if no image file is provided.

In [ ]:
IMAGE_PATH = None  # e.g. "sample_food.jpg" — set this to run the cell

if IMAGE_PATH and os.path.exists(IMAGE_PATH):
    with open(IMAGE_PATH, "rb") as f:
        image_bytes = f.read()
    mime = "image/jpeg" if IMAGE_PATH.lower().endswith((".jpg", ".jpeg")) else "image/png"
    analysis = analyze_food_image(image_bytes, mime)
    print(json.dumps(analysis, indent=2))
else:
    print("No IMAGE_PATH set — skipping the live image-analysis call. "
          "See api_server.py POST /api/image/analyze for the full HTTP flow.")

## Summary for the viva

- **Plan-act loop deciding the next step:** `app/agent.py`,
  `run_agent()`, the `for _ in range(MAX_AGENT_STEPS):` loop — each
  iteration sends the growing message history (including prior tool
  results) back to Groq and lets the model choose the next
  `tool_calls` or a final answer.
- **A real tool call:** `app/tools.py` — `lookup_calories`, `add_meal`,
  `remove_meal`, dispatched via `TOOL_FUNCTIONS` in `app/agent.py`
  based on the LLM's own `tool_calls` output.
- **Memory read back:** `app/memory.py` — `MemoryStore.get_state()`,
  read at the start of every `run_agent()` call and shown in the
  `read_memory` trace event above; Demo 2 and Demo 3 both show state
  logged in an earlier call being used in a later one.